### Dataset and Task Metadata

In [2]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="spotify_tracks_genre_prediction",
    dataset_year="2022",
    domain_str="technology & internet",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset",
    download_description="""
We download the data from Kaggle.

kaggle datasets download maharshipandya/-spotify-tracks-dataset &&  mv -- "-spotify-tracks-dataset.zip" spotify-tracks-dataset.zip && unzip spotify-tracks-dataset.zip && rm spotify-tracks-dataset.zip
mkdir -p local-data-warehouse/spotify_tracks_genre_prediction && mv dataset.csv local-data-warehouse/spotify_tracks_genre_prediction/
""",
    # References
    academic_reference_bibtex="""@misc{maharshi_pandya_2022,
	title={Spotify Tracks Dataset},
	url={https://www.kaggle.com/dsv/4372070},
	DOI={10.34740/KAGGLE/DSV/4372070},
	publisher={Kaggle},
	author={Maharshi Pandya},
	year={2022}
}
""",
    academic_reference_bibtex_key="maharshi_pandya_2022",
    license="Database: Open Database, Contents: © Original Authors",
    data_tags=["IID"],
    curation_comments="""
We start with the dataset.csv from kaggle.

- The data is balanced w.r.t the target column 'track_genre', as for each genre 1000 entries were scrapped.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="track_genre",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="track_gerne",
)

## Preprocessing

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "dataset.csv")
print("Loaded data shape:", df.shape)

Loaded data shape: (114000, 21)


## WARNING
We stopped here as the label track_genre does not make sense for the data. We might revisit this later or remove the dataset entirely.

In [6]:
df[df['track_id'].duplicated(keep=False)].sort_values(by="track_id").head(10)

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
15028,15028,001APMDOl3qtx1526T11n1,Pink Sweat$;Kirby,New RnB,Better,0,176320,False,0.613,0.471,1,-6.644,0,0.1070,0.31600,0.000001,0.1170,0.406,143.064,4,chill
103211,103211,001APMDOl3qtx1526T11n1,Pink Sweat$;Kirby,New RnB,Better,0,176320,False,0.613,0.471,1,-6.644,0,0.1070,0.31600,0.000001,0.1170,0.406,143.064,4,soul
85578,85578,001YQlnDSduXd5LgBd66gT,Soda Stereo,Soda Stereo (Remastered),El Tiempo Es Dinero - Remasterizado 2007,38,177266,False,0.554,0.921,2,-4.589,1,0.0758,0.01940,0.088100,0.3290,0.700,183.571,1,punk-rock
100420,100420,001YQlnDSduXd5LgBd66gT,Soda Stereo,Soda Stereo (Remastered),El Tiempo Es Dinero - Remasterizado 2007,38,177266,False,0.554,0.921,2,-4.589,1,0.0758,0.01940,0.088100,0.3290,0.700,183.571,1,ska
91801,91801,003vvx7Niy0yvhvHt4a68B,The Killers,Hot Fuss,Mr. Brightside,86,222973,False,0.352,0.911,1,-5.230,1,0.0747,0.00121,0.000000,0.0995,0.236,148.033,4,rock
3257,3257,003vvx7Niy0yvhvHt4a68B,The Killers,Hot Fuss,Mr. Brightside,86,222973,False,0.352,0.911,1,-5.230,1,0.0747,0.00121,0.000000,0.0995,0.236,148.033,4,alternative
2106,2106,003vvx7Niy0yvhvHt4a68B,The Killers,Hot Fuss,Mr. Brightside,86,222973,False,0.352,0.911,1,-5.230,1,0.0747,0.00121,0.000000,0.0995,0.236,148.033,4,alt-rock
33178,33178,004h8smbIoAkUNDJvVKwkG,Ouse;Powfu,Loners Diary,Lovemark,58,219482,True,0.808,0.331,5,-13.457,1,0.0557,0.13100,0.000000,0.2250,0.337,140.035,4,emo
94239,94239,004h8smbIoAkUNDJvVKwkG,Ouse;Powfu,Loners Diary,Lovemark,58,219482,True,0.808,0.331,5,-13.457,1,0.0557,0.13100,0.000000,0.2250,0.337,140.035,4,sad
97533,97533,006rHBBNLJMpQs8fRC2GDe,Calcinha Preta;Gusttavo Lima,CP 25 Anos (Ao Vivo em Aracaju),Agora Estou Sofrendo - Ao Vivo,47,260510,False,0.605,0.678,0,-3.257,1,0.0311,0.64200,0.000000,0.1570,0.439,125.059,4,sertanejo


## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_iid_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_iid_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

# The structure of splits is:
# splits = {
#     repeat_i: {
#         fold_i: (train_idx, test_idx),
#     }
# }

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [ ]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)